# StyleTTS 2

**Domain:** Speech & Audio  ·  **recommended addition**  ·  **runnable:** yes

A refresher on **StyleTTS 2** (Li et al., NeurIPS 2023) — the text-to-speech system that
reached *human-level* naturalness by combining **style diffusion** with **adversarial
training against a large speech language model**, while staying fast and non-autoregressive.

## 1. What & Why

**StyleTTS 2** is an end-to-end neural TTS model from Columbia University. On single-speaker
LJSpeech it **matched or beat ground-truth recordings on MOS naturalness**, and on multi-speaker
LibriTTS it set a new bar for zero-shot voice cloning — all while running far faster than
autoregressive, codec-token models like Bark or VALL-E.

Two ideas make it work:

- **Style diffusion.** Speech "style" (prosody, emotion, speaker timbre, delivery) is modeled
  as a **latent vector sampled by a diffusion model** conditioned on the text. So the model can
  *invent* a plausible, varied delivery for a sentence with **no reference audio at all** — or
  clone a voice when you *do* give it a reference clip.
- **SLM adversarial training.** A large pretrained **Speech Language Model (WavLM)** is used as
  the **discriminator**. Its rich, human-like speech representations push the generator toward
  natural output far more effectively than a from-scratch mel/waveform discriminator.

**Reach for StyleTTS 2 when** you want state-of-the-art naturalness and expressive, controllable
prosody, *and* you can run a PyTorch model (GPU nice, CPU works for short clips). **Skip it when**
you need on-device/embedded inference (use Piper), a turnkey hosted API (ElevenLabs), or built-in
nonverbal markup like `[laughs]` (use Bark) — StyleTTS 2 is a research codebase, not a product.

## 2. Mental Model

**Diffuse the *style*, not the *audio*.** Waveform diffusion (and codec-token LLMs like Bark)
is slow because the thing being generated is huge. StyleTTS 2's insight: the only part that
*needs* a rich, multi-modal generative prior is the small **style vector** — everything else can
be a fast, deterministic, non-autoregressive decoder. So it runs an expensive diffusion sampler
on a ~128-dim latent, then hands that style to a quick GAN vocoder.

```
                              ┌──────────────────────────┐
  text ─► phonemize ─► [PL-BERT text encoder] ─► content │
                              └──────────────────────────┘
                                         │
            (optional reference audio)   │   ┌──────────────────────────────┐
                       │                 │   │ STYLE DIFFUSION               │
                       ▼                 ▼   │ noise ─►…denoise…─► style s    │  ← the only
              [style encoder] ───────────────│ (prosody + timbre + emotion)  │    "creative" /
                                             └──────────────────────────────┘    diffused part
                                         │                 │
                                         ▼                 ▼
                            [duration + prosody predictors]  (s conditions these)
                                         │
                       differentiable upsampling (align phonemes → frames)
                                         │
                                         ▼
                            [decoder / HiFiGAN-iSTFT vocoder] ─► 24 kHz waveform
                                         │
                                         ▼
                  during TRAINING only:  [WavLM SLM discriminator]  ⇄ adversarial loss
```

The vocoder is a normal fast GAN. The "magic" is (a) the diffusion prior over the tiny style
latent, which gives diverse, expressive, optionally-cloned voices, and (b) a WavLM judge during
training that makes the output sound human.

## 3. Key Concepts

- **Style vector `s`.** A compact latent that captures *how* a sentence is said — prosody,
  rhythm, emotion, speaker identity. Conditions the duration predictor, prosody predictor, and
  decoder. StyleTTS 2 actually splits it into an **acoustic** style and a **prosodic** style.
- **Style diffusion.** An EDM-style diffusion model samples `s` from noise, conditioned on the
  text encoding (and a reference clip if cloning). This is what lets the model produce *diverse*
  deliveries and **zero-shot clone** from a few seconds of reference audio. `diffusion_steps`
  trades speed for style diversity; `embedding_scale` is its classifier-free-guidance knob.
- **SLM adversarial training.** A frozen, pretrained **WavLM** speech LM provides features for
  the discriminator. Training the generator to fool this human-trained judge is the single
  biggest naturalness win over StyleTTS 1.
- **Differentiable duration modeling.** Phoneme durations are turned into a **soft, monotonic
  alignment** (not a hard repeat), so gradients from the adversarial/waveform losses flow all
  the way back into the duration predictor — enabling true end-to-end training.
- **PL-BERT.** A **phoneme-level BERT** pretrained on phoneme sequences gives the text encoder
  prosody-aware representations, improving naturalness and out-of-domain robustness.
- **`alpha` / `beta`.** Inference blend factors: how much of the timbre (`alpha`) and prosody
  (`beta`) come from the *sampled* style vs. the *reference* audio. Higher = more diverse/less
  faithful to the reference.
- **Non-autoregressive + GAN vocoder.** Unlike Bark/VALL-E, decoding is one fast forward pass
  (HiFiGAN / iSTFTNet decoder), so it's real-time-capable on GPU.

## 4. Setup

The real model is a PyTorch codebase that needs **`espeak-ng`** (system package, for
phonemization) and downloads pretrained weights (hundreds of MB). The easiest wrapper is the
[`styletts2`](https://pypi.org/project/styletts2/) PyPI package; the reference implementation is
[`yl4579/StyleTTS2`](https://github.com/yl4579/StyleTTS2).

```bash
# System dependency (phonemizer backend):
sudo apt-get install espeak-ng        # Debian/Ubuntu
brew install espeak                   # macOS

# Python:
pip install styletts2                 # turnkey wrapper used in Example 3
# or clone yl4579/StyleTTS2 for training / full control

# The heavy synthesis cell is gated behind RUN_STYLETTS2 so this notebook still
# executes top-to-bottom without the weight download.
```

The two worked examples below run on **CPU with only NumPy** — they make the two core ideas
(style diffusion and differentiable duration alignment) concrete without any model download.

In [ ]:
import sys, numpy as np
print("Python", sys.version.split()[0], "| numpy", np.__version__)

# Optional deps for the gated real-synthesis cell — we only *report* availability here.
for mod in ("torch", "styletts2", "phonemizer", "librosa"):
    try:
        __import__(mod)
        print(f"  {mod:<11} available")
    except ImportError:
        print(f"  {mod:<11} not installed (fine — only needed for real synthesis)")

## 5. Worked Examples

The first two examples run on CPU with no download and make StyleTTS 2's two signature ideas
concrete: **(1)** style as a *diffusion-sampled latent* (why output is diverse and clonable),
and **(2)** *differentiable duration upsampling* (why it trains end-to-end). The third shows the
real synthesis call, gated so the notebook still runs.

### Example 1 — Style as a diffusion-sampled latent

StyleTTS 2 doesn't diffuse audio — it runs a diffusion sampler over a small **style vector** and
hands the result to a fast decoder. The toy below makes the *behavior* concrete: the learned
"style manifold" is two prosodic modes (say *calm* vs *excited*) in 2-D instead of 128-D, and a
Langevin sampler (a stand-in for the reverse diffusion process) lands on a **different point
every call**. That per-call diversity — with no reference audio — is exactly what style
diffusion buys you.

In [ ]:
import numpy as np

# The "style manifold" the diffusion model learned: two prosodic modes in 2-D.
centers = np.array([[-2.0, 0.0], [2.0, 1.0]])   # e.g. "calm" vs "excited" delivery

def score(x):
    # ∇ log p(x) for a Gaussian mixture: pull x toward the modes, weighted by responsibility.
    d = x - centers                              # (modes, dims)
    w = np.exp(-0.5 * (d ** 2).sum(1)); w /= w.sum()
    return -(w[:, None] * d).sum(0)

def sample_style(seed, steps=300, lr=0.05):
    # Annealed Langevin dynamics ≈ reverse diffusion sampling of the style vector.
    r = np.random.default_rng(seed)
    x = r.normal(size=2) * 3.0                   # start from noise
    for _ in range(steps):
        x = x + lr * score(x) + np.sqrt(2 * lr) * 0.25 * r.normal(size=2)
    return x

styles = np.array([sample_style(s) for s in range(6)])
print("Six sampled style vectors (each conditions a different prosody/voice):")
print(np.round(styles, 2))
nearest = np.abs(styles[:, None, :] - centers[None]).sum(2).argmin(1)
print("\nNearest mode per sample:", nearest, " (0=calm, 1=excited)")
print("No reference audio needed — every call samples a fresh, valid style.")

Each run lands on a different point on the manifold — and at inference you steer this: more
`diffusion_steps` explores it more, `embedding_scale` (classifier-free guidance) sharpens toward
the text-conditioned region, and passing a **reference clip** biases the sampler toward that
speaker's style — the zero-shot cloning path. The real model diffuses a ~128-dim vector, but the
"sample a style, then decode fast" principle is identical.

### Example 2 — Differentiable duration upsampling

After predicting how many frames each phoneme lasts, the model must **expand** phoneme encodings
to a frame sequence the decoder can vocode. A naive `repeat()` is non-differentiable w.r.t. the
durations, so gradients from the adversarial/waveform loss can't reach the duration predictor.
StyleTTS 2 uses a **soft, monotonic alignment** instead — below, each phoneme sits at its
cumulative center and frames are Gaussian-weighted around it. That smooth matrix is what makes
duration modeling trainable **end-to-end**.

In [ ]:
import numpy as np

phonemes  = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0]], float)  # 3 phonemes, 4-dim
durations = np.array([2.0, 3.0, 1.0])                                     # predicted frames each
T = int(durations.sum())

# Hard alignment: repeat each phoneme d times — correct, but NOT differentiable in d.
hard = np.repeat(phonemes, durations.astype(int), axis=0)

# Soft alignment: place each phoneme at its cumulative center, Gaussian-weight the frames.
centers = np.cumsum(durations) - durations / 2.0
frames  = np.arange(T) + 0.5
W = np.exp(-0.5 * ((frames[:, None] - centers[None, :]) / 0.7) ** 2)
W /= W.sum(1, keepdims=True)            # row-normalize → soft alignment matrix
soft = W @ phonemes                     # smooth frame-level features

print("durations:", durations, "-> total", T, "frames")
print("\nHard upsample (per-frame phoneme index):", hard.argmax(1))
print("\nSoft alignment matrix W  (frames x phonemes):")
print(np.round(W, 2))
print("\nW is differentiable in `durations`, so the adversarial loss can train the")
print("duration predictor end-to-end — a key StyleTTS 2 contribution.")

Notice `W` shares weight across neighboring phonemes at the boundaries — that smoothness is
precisely what gives a usable gradient. Nudge a duration up and the matrix (and every downstream
frame) shifts continuously, so the loss can say "this phoneme should be a little longer" and have
it stick.

### Example 3 — Real synthesis (gated behind `RUN_STYLETTS2`)

The actual call using the `styletts2` wrapper. It's skipped unless you set `RUN_STYLETTS2=1`,
because it needs `espeak-ng` and downloads pretrained weights. The code is correct as written —
note `diffusion_steps` (style diversity), `alpha`/`beta` (sampled-vs-reference blend), and the
optional `target_voice_path` for **zero-shot cloning**.

In [ ]:
import os

if os.getenv("RUN_STYLETTS2"):
    from styletts2 import tts                       # pip install styletts2 ; system espeak-ng
    model = tts.StyleTTS2()                          # downloads LJSpeech/LibriTTS weights once
    model.inference(
        "StyleTTS two reaches human level naturalness with style diffusion.",
        output_wav_file="styletts2_out.wav",
        # target_voice_path="reference.wav",         # zero-shot clone from a few seconds of audio
        diffusion_steps=10,                          # more steps -> more style diversity
        alpha=0.3, beta=0.7,                         # blend sampled style vs reference timbre/prosody
        embedding_scale=1.0,                         # classifier-free guidance strength
    )
    print("Wrote styletts2_out.wav")
else:
    print("RUN_STYLETTS2 not set — skipping the pretrained-weight download.")
    print("To run for real (needs system espeak-ng):")
    print("    pip install styletts2")
    print("    RUN_STYLETTS2=1 python ...")
    print("Call shape:")
    print("    from styletts2 import tts")
    print("    model = tts.StyleTTS2()")
    print('    model.inference("hello world", output_wav_file="out.wav", diffusion_steps=10)')

## 6. Gotchas & Pitfalls

- **`espeak-ng` is a hard dependency.** Phonemization happens through it; without the *system*
  package you get cryptic phonemizer errors. Install it before `pip install styletts2`.
- **Style diffusion is stochastic.** Different `diffusion_steps`/seeds give different prosody for
  the same text — great for variety, but don't assert byte-equal output in tests. For a stable
  voice, pin a reference clip and lower `alpha`/`beta`.
- **`alpha`/`beta` trade faithfulness for expressiveness.** Cloning a voice? Keep them low so the
  output tracks the reference. Want lively, varied delivery? Raise them and add diffusion steps —
  at the cost of drifting from the target timbre.
- **Long text needs splitting.** Like most non-autoregressive TTS, quality degrades on very long
  inputs; synthesize sentence-by-sentence and concatenate, keeping the same style vector for
  consistency.
- **Research code, rough edges.** The reference repo pins older Torch/dependency versions and
  expects phonemized input; expect environment friction versus a packaged product.
- **Not a `[laughs]`-markup model.** It controls style via vectors/reference audio, not inline
  tags — if you want laughter/SFX tokens, that's Bark's lane, not this.
- **Multi-speaker vs single-speaker checkpoints differ.** The LJSpeech model is one voice; zero-
  shot cloning needs the LibriTTS multispeaker checkpoint and a reference clip.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs. StyleTTS 2 |
|---|---|---|
| **StyleTTS 2** | SOTA naturalness, expressive/controllable prosody, zero-shot cloning, fast non-AR inference | Research codebase; `espeak-ng` + weight download; stochastic output |
| **VITS / [notebook](vits.ipynb)** | Fast, solid end-to-end neural TTS for a fixed voice | Lower naturalness ceiling; less style control / no diffusion prior |
| **Bark / [notebook](bark.ipynb)** | Nonverbal sounds (`[laughs]`), music, multilingual, quick demos | Slow, ~13 s cap, codec-token LLM; lower fidelity, no fast vocoder |
| **Coqui XTTS / [notebook](coqui-tts.ipynb)** | Turnkey zero-shot cloning across many languages | More packaged; StyleTTS 2 generally edges it on naturalness |
| **Piper / [notebook](piper-tts.ipynb)** | On-device, real-time, tiny footprint (Raspberry Pi) | Robotic by comparison; no style/emotion control |
| **ElevenLabs / [notebook](elevenlabs.ipynb)** | Best hosted quality, cloning, low latency, no infra | Paid SaaS, closed, data leaves your box |
| **Tacotron 2 / [notebook](tacotron.ipynb)** | Studying the classic mel + separate-vocoder pipeline | Needs a vocoder; dated naturalness; autoregressive |

**Rule of thumb:** want the most *natural, expressive, self-hosted* open TTS and can tolerate a
research-grade setup → StyleTTS 2. Need embedded/real-time → Piper. Need an API and zero ops →
ElevenLabs. Need laughter/SFX baked in → Bark.

## 8. Resources

- **StyleTTS 2 paper — "Towards Human-Level Text-to-Speech through Style Diffusion and
  Adversarial Training with Large Speech Language Models" (Li et al., NeurIPS 2023)**:
  https://arxiv.org/abs/2306.07691
- **Reference implementation (yl4579/StyleTTS2)** — training + inference code, checkpoints,
  Colab demos: https://github.com/yl4579/StyleTTS2
- **`styletts2` PyPI wrapper** — the turnkey `tts.StyleTTS2().inference(...)` API used above:
  https://pypi.org/project/styletts2/
- **Audio samples / demo page** — hear the human-level claims for yourself:
  https://styletts2.github.io/
- **PL-BERT (phoneme-level BERT)** — the text encoder pretraining that boosts prosody:
  https://github.com/yl4579/PL-BERT
- **WavLM paper (Chen et al., 2021)** — the speech LM used as the adversarial discriminator:
  https://arxiv.org/abs/2110.13900

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def hard_upsample(phonemes, durations):
    ...


def soft_alignment(durations, sigma=0.7):
    ...


def blend_style(sampled, reference, alpha):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE